# Structured output checks

Open this project folder in VS Code and select the shared Python kernel. The main lesson loads six made-up replies and checks them with rules we wrote. For example, the confidence score must be a number. The final cell can get a fresh reply from a model and check it the same way. That cell is off by default.

In [ ]:
import json
from pathlib import Path
from pprint import pprint

fixtures = json.loads(Path("docs/dataset/responses.json").read_text())
pprint([(item["id"], item["kind"]) for item in fixtures])

## The contract helper

`tools/validate.py` defines `classification_schema` and `validate_result`. The latter collects `iter_errors`, keeps the raw dictionary, and withholds invalid objects. Required fields, types, categories, and confidence bounds are structural checks. Import them after reading that small file.

In [ ]:
from tools.validate import classification_schema, validate_result

pprint(classification_schema(min_confidence=0.5))

## Normal, problematic, and edge fixtures

Inspect each field/rule failure. The edge fixture lies exactly at the inclusive minimum. `raw` is inspection data; only `exposed` would reach the application user.

In [ ]:
baseline = {item["id"]: validate_result(item["raw"], 0.5) for item in fixtures}
for name, result in baseline.items():
    print(name)
    pprint(result)

## A valid structure can be false

The fictional source policy grants **20 days** of annual leave. Our 200-day response passes because this checker knows nothing about policy evidence. Its confidence is also self-reported.

In [ ]:
pprint(baseline["wrong_fact"])
assert baseline["wrong_fact"]["decision"] == "allow"
assert baseline["edge"]["decision"] == "allow"

## Change one setting on the same inputs

Raise only `min_confidence` from 0.5 to 0.8. The same edge response becomes blocked; the wrong fact still passes. This stricter contract does not establish accuracy.

In [ ]:
candidate = {item["id"]: validate_result(item["raw"], 0.8) for item in fixtures}
for item in fixtures:
    name = item["id"]
    pprint({"id": name, "baseline": baseline[name]["decision"],
            "candidate": candidate[name]["decision"], "candidate_errors": candidate[name]["errors"]})
assert candidate["edge"]["decision"] == "block"
assert candidate["wrong_fact"]["decision"] == "allow"

## Optional actual model response — disabled

`models/groq_model.py` loads the ignored root `.env` only inside `classify_live`, uses the actual prompt in `prompts/classify.txt`, and obtains a dictionary through the provider JSON helper. Read that helper before enabling it. The default branch makes zero API calls and reads no credentials. Enabling requires your Groq key and an available model; provider errors stay visible. No live result was obtained during preparation.

In [ ]:
RUN_LIVE = False
if RUN_LIVE:
    from models.groq_model import classify_live
    live_object = classify_live("How much annual leave do I have?", classification_schema())
    pprint({"provenance": "actual Groq model output", "validation": validate_result(live_object)})
else:
    print("Live step skipped: zero model/API calls; fixture checks only.")